# MCP for scientific agents - Colab quickstart (Section 8)

Self-contained: no repo clone needed. Uses an **in-process** MCP server (`Client(server)`), so there is no subprocess to manage.

**Before running:** Colab sidebar -> key icon -> add secrets `LLM_API_KEY` (and optionally `LLM_BASE_URL`, `MODEL`), toggle *Notebook access* on.

In [ ]:
%pip install -q 'mcp>=2,<3' openai

In [ ]:
import os
from google.colab import userdata

os.environ['LLM_API_KEY'] = userdata.get('LLM_API_KEY')
def _opt(name, default):
    try: return userdata.get(name)
    except Exception: return default
BASE_URL = _opt('LLM_BASE_URL', 'https://api.openai.com/v1') # Optional: override the default base URL with your own OpenAI-compatible endpoint (e.g. OpenRouter, Anthropic, Groq, vLLM, Ollama, LiteLLM, etc.)
MODEL = _opt('MODEL', 'gpt-4o-mini')
print(BASE_URL, MODEL)

## 1. A tiny scientific MCP server (SDK v2: `MCPServer`)

In [ ]:
import math
from mcp.server.mcpserver import MCPServer
from mcp.server.mcpserver.exceptions import ToolError

server = MCPServer('materials-mini')

@server.tool()
def scherrer_crystallite_size(fwhm_deg: float, two_theta_deg: float, wavelength_A: float = 1.5406, K: float = 0.9) -> dict:
    """Scherrer crystallite size in nm from peak FWHM (degrees 2-theta). Instrument broadening already removed."""
    if fwhm_deg <= 0 or not 0 < two_theta_deg < 180:
        raise ToolError('fwhm_deg must be > 0 and two_theta_deg in (0, 180)')
    beta, theta = math.radians(fwhm_deg), math.radians(two_theta_deg / 2)
    return {'value': round(K * wavelength_A / (beta * math.cos(theta)) / 10, 3), 'units': 'nm', 'method': 'Scherrer'}

@server.tool()
def bragg_d_spacing(two_theta_deg: float, wavelength_A: float = 1.5406) -> dict:
    """d-spacing in angstrom from a diffraction peak position (Bragg's law, first order)."""
    if not 0 < two_theta_deg < 180:
        raise ToolError('two_theta_deg must be in (0, 180)')
    return {'value': round(wavelength_A / (2 * math.sin(math.radians(two_theta_deg / 2))), 4), 'units': 'angstrom', 'method': 'Bragg'}

## 2. Connect in-process and list tools

Colab already runs an event loop, so use top-level `await` (no `asyncio.run`).

In [ ]:
from mcp import Client

async with Client(server) as client:
    for t in (await client.list_tools()).tools:
        print(t.name, '-', t.description)
    r = await client.call_tool('bragg_d_spacing', {'two_theta_deg': 38.47})
    print(r.content[0].text)

## 3. Agent loop against any OpenAI-compatible endpoint

In [ ]:
import json
from openai import AsyncOpenAI

llm = AsyncOpenAI(api_key=os.environ['OPENAI_API_KEY'], base_url=BASE_URL)

async def ask(question):
    async with Client(server) as client:
        tools = [{'type': 'function', 'function': {'name': t.name, 'description': t.description or '', 'parameters': t.input_schema}}
                 for t in (await client.list_tools()).tools]
        messages = [{'role': 'system', 'content': 'Use the tools for every calculation. Quote units.'}, {'role': 'user', 'content': question}]
        for _ in range(6):
            msg = (await llm.chat.completions.create(model=MODEL, messages=messages, tools=tools)).choices[0].message
            messages.append(msg.model_dump(exclude_none=True))
            if not msg.tool_calls:
                return msg.content
            for call in msg.tool_calls:
                res = await client.call_tool(call.function.name, json.loads(call.function.arguments))
                print('tool:', call.function.name, call.function.arguments)
                messages.append({'role': 'tool', 'tool_call_id': call.id, 'content': res.content[0].text})

print(await ask('XRD peak at 2-theta 38.47 deg with FWHM 0.30 deg (Cu Ka). Give d-spacing and crystallite size.'))

## Exercise
Add a `vegard_lattice_parameter(a_A, a_B, fraction_B)` tool, re-run `ask(...)` with a question about a 30% B alloy.

**Colab notes:** secrets replace `.env`; `stdio` servers also work (`!python server.py`), but in-process is simpler; sessions are saved, so re-run the install cell after a runtime reset.